# Data Understanding & Preprocessing Final — Indra

Notebook ini merupakan versi final terstandar untuk proyek **Segmentasi UMKM Lokal Kota Bandung**.

Perbaikan utama:
- parser CSV mencatat baris bermasalah;
- metadata run dan sumber file dipertahankan;
- unit awal disebut **listing Google Maps**, bukan langsung UMKM;
- validasi rating dan jumlah ulasan eksplisit;
- deduplikasi memakai `entity_key`;
- `street` kosong menggunakan URL sebagai identitas cadangan dan dikarantina jika tetap ambigu;
- filter Kota Bandung lebih ketat;
- audit brand/non-UMKM memakai keputusan manual, exact list lama, pola kuat, dan daftar kandidat review;
- `reviewId` dipakai untuk deduplikasi ulasan;
- title multi-listing tidak dipetakan secara paksa;
- seluruh data yang dibuang masuk sheet audit;
- menghasilkan tabel rekonsiliasi otomatis.

> Kategori seperti **hotel, klinik, dealer, dan supermarket tidak otomatis dibuang** hanya berdasarkan kategori. Kategori tersebut masuk `REVIEW`, karena kategori saja tidak membuktikan skala usaha.


In [ ]:
import os
import re
import hashlib
import unicodedata
from urllib.parse import unquote

import numpy as np
import pandas as pd
from google.colab import files

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil dimuat.")

In [ ]:
# ============================================================
# KONFIGURASI NOTEBOOK — INDRA
# ============================================================

NAMA_ANGGOTA = 'Indra'
ENABLE_ROW_SHIFT = False

# Tanggal yang tidak dapat dibuktikan tidak diisi atau ditebak.
CONFIG_RUNS = [{'run_id': 'INDRA_R1',
  'crawl_date': None,
  'info_file': 'data_umkm_apify_scraper_no_text.csv',
  'review_file': 'data_umkm_apify_scraper_ada_text_nya.csv'},
 {'run_id': 'INDRA_R2',
  'crawl_date': None,
  'info_file': 'data_umkm_apify_scraper_2_no_text.csv',
  'review_file': 'data_umkm_apify_scraper_2_ada_text.csv'}]

OUTPUT_CSV_AUDIT = '01_Hasil_Preprocessing_Indra_Final.csv'
OUTPUT_CSV = '01_Hasil_Preprocessing_Indra_Final_Model_7Kolom.csv'

print(f"Notebook: {NAMA_ANGGOTA}")
print(f"Jumlah run: {len(CONFIG_RUNS)}")
print(f"Koreksi row-shift aktif: {ENABLE_ROW_SHIFT}")

## Kebijakan audit brand dan entitas

`Pasar Baru` dan `Sudirman Street` tidak lagi masuk whitelist. Daftar lokal lama hanya menjadi petunjuk, bukan jaminan otomatis. Keputusan manual paling aman dilakukan berdasarkan `entity_key`, agar cabang bernama sama tidak ikut terkena keputusan yang salah.

In [ ]:
# ============================================================
# KEBIJAKAN AUDIT ENTITAS — MASTER KELOMPOK
# ============================================================
# Prinsip:
# 1. Keputusan manual berbasis entity_key/title mempunyai prioritas tertinggi.
# 2. Daftar exact lama tetap dipakai.
# 3. Brand/jaringan yang sangat jelas dapat di-drop otomatis.
# 4. Kategori/nama yang ambigu hanya diberi status REVIEW, bukan langsung di-drop.
# 5. Daftar nama lokal lama hanya menjadi PETUNJUK, bukan whitelist mutlak.
#    'Pasar Baru' dan 'Sudirman Street' sengaja TIDAK dimasukkan sebagai petunjuk lokal,
#    karena keduanya adalah entitas agregat, bukan satu usaha tunggal.

MANUAL_ENTITY_OVERRIDES = {
    # Isi setelah melihat sheet Audit_Entitas, contoh:
    # "nama normal || alamat normal": {
    #     "status": "UMKM",  # UMKM / BUKAN_UMKM / PERLU_TINJAU
    #     "alasan": "Hasil verifikasi dua anggota kelompok",
    #     "sumber": "Catatan verifikasi kelompok"
    # }
}

MANUAL_TITLE_DECISIONS = {'Plataran Bandung': {'status': 'BUKAN_UMKM',
                      'alasan': 'Venue dining dan event milik grup hospitality nasional Plataran Indonesia; bukan '
                                'usaha independen tunggal.',
                      'sumber': 'Verifikasi manual kelompok: situs Plataran dan sumber publik terkait.'},
 'Sadrasa Kitchen & Bar': {'status': 'BUKAN_UMKM',
                           'alasan': 'Restoran resmi di Hotel Pullman Bandung Grand Central dan merupakan bagian '
                                     'jaringan hotel Accor.',
                           'sumber': 'Verifikasi manual kelompok: situs resmi Pullman/Accor.'},
 'Mad Cow Wine & Grill Bandung': {'status': 'BUKAN_UMKM',
                                  'alasan': 'Unit F&B resmi Hotel Pullman Bandung Grand Central.',
                                  'sumber': 'Verifikasi manual kelompok: situs resmi hotel dan akun resmi.'},
 'Restoran Simpang Raya Bandung': {'status': 'BUKAN_UMKM',
                                   'alasan': 'Bagian jaringan rumah makan nasional dengan cabang lintas kota dan '
                                             'provinsi.',
                                   'sumber': 'Verifikasi manual kelompok: situs jaringan Simpang Raya.'},
 'Madame Sari Restaurant': {'status': 'BUKAN_UMKM',
                            'alasan': 'Dioperasikan oleh grup Kartika Sari dan bukan usaha independen tunggal.',
                            'sumber': 'Verifikasi manual kelompok: GoFood dan sumber publik terkait.'},
 'Bakso Rusuk Al Fatih Bandung': {'status': 'UMKM',
                                  'alasan': 'Usaha independen menurut verifikasi kelompok.',
                                  'sumber': '-'},
 'Singgalang Jaya': {'status': 'UMKM', 'alasan': 'Rumah makan independen menurut verifikasi kelompok.', 'sumber': '-'},
 'Lontong Sayur Padang Dipati Ukur Bandung': {'status': 'UMKM',
                                              'alasan': 'Warung independen menurut verifikasi kelompok.',
                                              'sumber': '-'},
 'Bakso Rusuk Samanhudi Dago': {'status': 'UMKM',
                                'alasan': 'Warung independen menurut verifikasi kelompok.',
                                'sumber': '-'},
 'Oleh Oleh Bandung | Batagor Coy | Kuliner Bandung Paling Dicari': {'status': 'UMKM',
                                                                     'alasan': 'Terverifikasi sebagai usaha tunggal '
                                                                               'Batagor Coy; nama panjang merupakan '
                                                                               'optimasi kata kunci listing.',
                                                                     'sumber': 'Verifikasi manual kelompok: situs dan '
                                                                               'akun usaha.'},
 'Kantin CAB (Cuanki Asli Bandung)': {'status': 'UMKM',
                                      'alasan': 'Kantin/warung independen menurut verifikasi kelompok.',
                                      'sumber': '-'},
 'Sate Maranggi Haji Yetty Cabang Bandung': {'status': 'PERLU_TINJAU',
                                             'alasan': 'Usaha keluarga dengan jaringan lintas kota/provinsi; keputusan '
                                                       'akhir perlu disepakati berdasarkan definisi operasional '
                                                       'penelitian.',
                                             'sumber': 'Verifikasi manual kelompok.'},
 'Padang Merdeka Bandung': {'status': 'UMKM',
                            'alasan': 'Rumah makan independen menurut verifikasi kelompok.',
                            'sumber': '-'},
 'Sate Maulana Yusuf': {'status': 'UMKM',
                        'alasan': 'Warung sate independen tunggal menurut verifikasi kelompok.',
                        'sumber': 'Verifikasi manual kelompok.'},
 'Zona Merah (Zomer) Sukajadi': {'status': 'UMKM',
                                 'alasan': 'Multi-gerai terbatas di Bandung dan dipertahankan berdasarkan keputusan '
                                           'operasional kelompok.',
                                 'sumber': 'Verifikasi manual kelompok.'},
 'Kedai Mie Dago': {'status': 'UMKM', 'alasan': 'Kedai independen menurut verifikasi kelompok.', 'sumber': '-'},
 "Sam's Strawberry Corner": {'status': 'UMKM',
                             'alasan': 'Kafe/kedai independen menurut verifikasi kelompok.',
                             'sumber': '-'}}

# Daftar exact dari notebook lama. Pencocokan dilakukan pada nama yang sudah dinormalisasi.
EXACT_DROP_LAMA = {'Baltos': 'Nama gedung pusat perbelanjaan, bukan 1 UMKM tunggal',
 'Pasar Baru': 'Nama pasar/gedung, bukan 1 UMKM tunggal',
 'Plaza Parahyangan': 'Nama gedung pusat perbelanjaan',
 'Plaza parahyangan': 'Nama gedung pusat perbelanjaan (varian penulisan)',
 'Grutty Plaza': 'Nama gedung pusat perbelanjaan',
 'Cibaduyut Plaza': 'Nama gedung pusat perbelanjaan',
 'Kosambi Plaza': 'Nama gedung pusat perbelanjaan',
 'Pasar Kaget Gasibu': 'Nama pasar/gedung, bukan 1 UMKM tunggal',
 'Pasar Lilin': 'Nama pasar/gedung, bukan 1 UMKM tunggal',
 'Gramedia Merdeka Bandung': 'Jaringan toko buku nasional (Kompas Gramedia)',
 'Gramedia Bandung Festival Citylink': 'Jaringan toko buku nasional (Kompas Gramedia)',
 'Rumah Mode Factory Outlet': 'Factory outlet skala besar dikenal luas',
 'Heritage Factory Outlet Bandung': 'Factory outlet skala besar dikenal luas',
 'MOC 23Paskal': 'Metro Outlet Cibaduyut, factory outlet skala besar',
 'LOTTE GROSIR BANDUNG': 'Jaringan grosir Lotte, korporasi ritel besar',
 'EIGER Adventure Flagship Store Sumatera Bandung': 'Brand nasional EIGER',
 'EIGER Adventure Flagship Store Cihampelas Bandung': 'Brand nasional EIGER',
 'EIGER Adventure Store Setiabudhi Bandung': 'Brand nasional EIGER',
 'EIGER Adventure Store Ujung Berung Bandung': 'Brand nasional EIGER',
 'EIGER Adventure Store - Mall - Bandung Indah Plaza (BIP)': 'Brand nasional EIGER',
 'EIGER ADVENTURE STORE PELAJAR PEJUANG BANDUNG': 'Brand nasional EIGER',
 'EIGER Adventure Store Buah batu Bandung': 'Brand nasional EIGER',
 'Eiger Adventure Store Suci': 'Brand nasional EIGER',
 'Arei Flagship Experience Store Cihampelas': 'Brand nasional Arei (Eiger Group)',
 'UNIQLO • Paskal 23': 'Brand ritel internasional (Fast Retailing Jepang)',
 'Nike Factory Store': 'Brand internasional Nike',
 'Rabbani': 'Brand fashion muslim skala nasional',
 'Rabbani Buah Batu': 'Brand fashion muslim skala nasional',
 'SPORTS STATION TRUNOJOYO': 'Jaringan ritel olahraga nasional (MAP Group)',
 'Sports Station • TSM Bandung': 'Jaringan ritel olahraga nasional (MAP Group)',
 'Batik Keris Grand Yogya Kepatihan': 'Brand batik skala nasional',
 'Sociolla Store • 23 PasKal Shopping Center': 'Jaringan omnichannel kecantikan nasional (60+ toko, Social Bella)',
 'Erafone Bandung Electronic Center (UG-A03)': 'Jaringan ritel nasional (PT Erajaya Swasembada Tbk, 1.100+ toko)',
 'Informa Electronics BEC': 'Jaringan furnitur/elektronik nasional (Kawan Lama Group)',
 'TOYS KINGDOM KINGS SHOPPING CENTER': 'Jaringan mainan nasional (Kawan Lama Group)',
 'Toys Kingdom Living Plaza Pasir Kaliki': 'Jaringan mainan nasional (Kawan Lama Group)',
 'Toys Kingdom Mall Citylink Bandung': 'Jaringan mainan nasional (Kawan Lama Group)',
 'Cargo Factory Outlet': 'Factory outlet skala besar dikenal luas',
 'Anakecil Factory Outlet': 'Factory outlet skala besar dikenal luas',
 'Log In Megastore ABC': 'Ritel elektronik skala besar, bukan UMKM',
 'Martha Tilaar Salon & Day Spa': 'Jaringan salon & spa kecantikan skala nasional (Martha Tilaar Group)',
 'Erajaya Grafindo Bandung': 'Bagian dari PT Erajaya Swasembada Tbk, korporasi ritel elektronik nasional',
 'Alfamart Griya Cempaka Arum': 'Jaringan minimarket ritel nasional (PT Sumber Alfaria Trijaya Tbk)',
 'Point Cafe Pasteur 78': 'Sub-brand kedai kopi resmi Indomaret Point, bukan cafe independen -- dikonfirmasi eksplisit '
                          "dari isi ulasan pelanggan ('Point Cafe adalah cafe yang merupakan bagian dari convenient "
                          "store, Indomaret Point')",
 'Point Cafe Dipatiukur': "Sub-brand kedai kopi resmi Indomaret Point (alamat tercatat 'Indomaret Point, Jl. Dipati "
                          "Ukur No.85')",
 'Point Cafe Cihampelas 198': 'Sub-brand kedai kopi resmi Indomaret Point',
 'Point Cafe Ciumbuleuit': 'Sub-brand kedai kopi resmi Indomaret Point',
 'Starbucks PVJ': 'Waralaba kedai kopi internasional (Starbucks Corporation)',
 'Starbucks Coffee - Ciwalk Bandung': 'Waralaba kedai kopi internasional',
 'KFC Setiabudi Bandung': 'Waralaba restoran cepat saji internasional (Yum! Brands)',
 "McDonald's Setiabudi Bandung": 'Waralaba restoran cepat saji internasional',
 'Pizza Hut Delivery - PHD Indonesia': 'Waralaba pizza internasional (Yum! Brands)',
 'Pizza Hut Restoran': 'Waralaba pizza internasional (2 lokasi berbeda pada data ini)',
 'Pepper Lunch': 'Jaringan restoran cepat saji internasional (Pepper Food Service)',
 'Yoshinoya Ciwalk Bandung': 'Waralaba restoran cepat saji Jepang skala internasional',
 'Mie Gacoan Dipatiukur': 'Waralaba mie pedas nasional (PT Pesta Pora Abadi, 260+ cabang, dikelola korporasi pusat)',
 'Mie Gacoan Dago': 'Waralaba mie pedas nasional (PT Pesta Pora Abadi, 260+ cabang)',
 'Mie Gacoan Bandung - Paskal': 'Waralaba mie pedas nasional (PT Pesta Pora Abadi, 260+ cabang)',
 'Mie Gacoan Bandung - Setiabudi': 'Waralaba mie pedas nasional (PT Pesta Pora Abadi, 260+ cabang)',
 'Wingstop - Cihampelas Bandung': 'Waralaba restoran cepat saji internasional (sayap ayam)',
 'Fore Coffee - Paris Van Java, Bandung': 'Waralaba kedai kopi nasional (Fore Coffee, puluhan cabang)',
 'Fore Coffee - Cihampelas Walk, Bandung': 'Waralaba kedai kopi nasional (Fore Coffee)',
 'Warunk Upnormal Sumur Bandung': 'Waralaba kafe nasional (puluhan cabang di seluruh Indonesia)',
 'Kopi Nako Bandung': 'Brand kopi dengan 39-48 cabang nasional, dikelola 1 entitas pusat -- CATATAN: sejumlah artikel '
                      "media menyebut brand ini masih dikategorikan 'UMKM' oleh publik meski skala cabangnya besar; "
                      'dimasukkan ke daftar drop karena pola operasionalnya (banyak cabang seragam, brand terpusat) '
                      'lebih dekat ke waralaba korporat daripada 1 UMKM tunggal -- keputusan ini bisa didiskusikan '
                      'ulang oleh kelompok jika ingin argumen berbeda',
 'Restoran Sederhana Setiabudi Bandung': 'Waralaba Restoran Padang nasional (RM Sederhana Group, ratusan cabang) -- '
                                         "BUKAN sama dengan 'Warteg Sederhana' atau 'Nasi Goreng Sederhana' pada data "
                                         'ini yang merupakan warung independen bernama umum',
 'Restoran Sederhana Pasteur': 'Waralaba Restoran Padang nasional (RM Sederhana Group)',
 'Restoran Sederhana Masakan Padang': 'Waralaba Restoran Padang nasional (RM Sederhana Group)',
 'Bebek Kaleyo Bandung': 'Waralaba restoran bebek nasional (puluhan cabang di berbagai kota)',
 'Sushi Tei Flamboyant': 'Waralaba restoran sushi skala nasional/regional'}

# Kategori yang cukup kuat untuk dikeluarkan otomatis karena merupakan institusi,
# fasilitas publik, korporasi, atau entitas agregat.
KATEGORI_AUTO_DROP = ['Rumah Sakit',
 'Rumah Sakit Umum',
 'Rumah Sakit Swasta',
 'Rumah Sakit Bersalin',
 'Pusat Kesehatan Masyarakat',
 'Puskesmas',
 'Pusat Perbelanjaan',
 'Bank',
 'Asuransi',
 'Kantor Perusahaan',
 'Pabrik',
 'Universitas',
 'Sekolah',
 'Terminal',
 'Stasiun Kereta Api',
 'Kantor Pemerintah',
 'Institusi Publik']

# Kategori berikut TIDAK boleh langsung dianggap bukan UMKM.
# Hotel, klinik, dan dealer dapat saja dimiliki usaha kecil/menengah.
# Karena itu statusnya REVIEW dan harus diperiksa pada sheet Audit_Entitas.
KATEGORI_PERLU_TINJAU = ['Hotel',
 'Toserba',
 'Dealer Sepeda Motor',
 'Klinik Gigi',
 'Klinik Medis',
 'Dealer Mobil',
 'Dealer Honda',
 'Layanan Transportasi',
 'Supermarket',
 'Pasar',
 'Gedung',
 'Food Court',
 'Apartemen']

# Pola brand/jaringan yang cukup kuat untuk auto-drop.
POLA_AUTO_DROP = {'\\bstarbucks\\b': 'Jaringan kedai kopi internasional',
 '\\bmcdonald(?:s)?\\b': 'Jaringan restoran cepat saji internasional',
 '\\bkfc\\b': 'Jaringan restoran cepat saji internasional',
 '\\bburger king\\b': 'Jaringan restoran cepat saji internasional',
 '\\bpizza hut\\b|\\bphd indonesia\\b': 'Jaringan pizza internasional',
 '\\bhokben\\b': 'Jaringan restoran nasional',
 '\\bmixue\\b': 'Jaringan waralaba',
 '\\bj\\s*co\\b': 'Jaringan waralaba',
 '\\bchatime\\b': 'Jaringan waralaba',
 '\\bmie gacoan\\b|\\bgacoan\\b': 'Jaringan kuliner nasional',
 '\\bfore coffee\\b': 'Jaringan kedai kopi nasional',
 '\\bpoint cafe\\b': 'Sub-brand Indomaret Point',
 '\\bwingstop\\b': 'Jaringan restoran internasional',
 '\\byoshinoya\\b': 'Jaringan restoran internasional',
 '\\bpepper lunch\\b': 'Jaringan restoran internasional',
 '\\bsushi tei\\b': 'Jaringan restoran',
 '\\bwarunk upnormal\\b': 'Jaringan kafe nasional',
 '\\bbebek kaleyo\\b': 'Jaringan restoran nasional',
 '\\brestoran sederhana\\b': 'Jaringan rumah makan nasional',
 '\\bkopi nako\\b': 'Jaringan kedai kopi multi-cabang',
 '\\beiger\\b': 'Brand ritel nasional',
 '\\buniqlo\\b': 'Brand ritel internasional',
 '\\bh m\\b': 'Brand ritel internasional H&M',
 '\\bzara\\b': 'Brand ritel internasional',
 '\\brabbani\\b': 'Brand fashion nasional',
 '\\bsociolla\\b': 'Jaringan kecantikan nasional',
 '\\berafone\\b|\\berajaya\\b': 'Jaringan ritel elektronik nasional',
 '\\binforma\\b': 'Jaringan ritel nasional',
 '\\bace hardware\\b': 'Jaringan ritel nasional',
 '\\bmr diy\\b': 'Jaringan ritel nasional',
 '\\bikea\\b': 'Jaringan ritel internasional',
 '\\bibox\\b': 'Jaringan ritel elektronik',
 '\\bgramedia\\b': 'Jaringan toko buku nasional',
 '\\bsports station\\b': 'Jaringan ritel olahraga nasional',
 '\\btoys kingdom\\b': 'Jaringan ritel nasional',
 '\\blotte\\b': 'Jaringan ritel besar',
 '\\bhypermart\\b': 'Jaringan ritel besar',
 '\\bsuperindo\\b': 'Jaringan supermarket nasional',
 '\\bindomaret\\b': 'Jaringan minimarket nasional',
 '\\balfamart\\b': 'Jaringan minimarket nasional',
 '\\balfamidi\\b': 'Jaringan minimarket nasional',
 '\\blawson\\b': 'Jaringan convenience store',
 '\\bcircle k\\b': 'Jaringan convenience store',
 '\\bplanet ban\\b': 'Jaringan bengkel dan ban nasional',
 '\\bkimia farma\\b': 'Jaringan apotek nasional/BUMN',
 '\\bapotek k ?24\\b|\\bapotek k24\\b': 'Jaringan apotek waralaba nasional',
 '\\belectronic city\\b': 'Jaringan ritel elektronik nasional',
 '\\bsamsung electronics indonesia\\b': 'Perusahaan elektronik besar',
 '\\bastra motor\\b|\\bauto2000\\b|\\btunas toyota\\b': 'Jaringan dealer korporasi',
 '\\bshop bike\\b': 'Jaringan ritel/bengkel nasional',
 '\\bwuling maju motor\\b': 'Dealer resmi jaringan besar',
 '\\byamaha .*official\\b|\\byamaha flagship\\b': 'Dealer/flagship resmi jaringan besar',
 '\\bmartha tilaar\\b': 'Jaringan kecantikan nasional',
 '\\bbatik keris\\b': 'Brand batik nasional',
 '\\bnike factory store\\b': 'Brand internasional',
 '\\brumah mode factory outlet\\b|\\bheritage factory outlet\\b': 'Factory outlet skala besar',
 '^suzuki setiabudi bandung pt nusantara jaya sentosa$': 'Dealer resmi berskala perusahaan',
 '^wijaya toyota dago$': 'Dealer resmi jaringan besar',
 '^wuling maju motor bandung$': 'Dealer resmi jaringan besar',
 '^yamaha jg bandung official$': 'Dealer resmi jaringan besar',
 '^yamaha flagship shop bandung official$': 'Flagship store resmi jaringan besar'}

# Pola ambigu: hanya diberi flag REVIEW.
POLA_PERLU_TINJAU = {'\\bahass\\b': 'AHASS dapat dimiliki usaha lokal independen atau jaringan; perlu verifikasi',
 '\\bdealer\\b': 'Dealer perlu verifikasi skala dan afiliasi',
 '\\bservice center\\b': 'Service center perlu verifikasi afiliasi',
 '\\bmall\\b|\\bplaza\\b|\\btrade center\\b|\\bshopping center\\b|\\btown square\\b|\\bsquare\\b': 'Kemungkinan pusat '
                                                                                                   'belanja atau '
                                                                                                   'tenant yang '
                                                                                                   'memakai nama '
                                                                                                   'lokasi',
 '\\bpasar\\b|\\bmarket\\b': 'Kemungkinan entitas agregat atau usaha individual yang memakai kata pasar/market',
 '\\bhotel\\b|\\bresort\\b': 'Hotel dapat berupa usaha lokal atau jaringan besar',
 '\\brumah sakit\\b|\\bpuskesmas\\b': 'Institusi kesehatan, bukan unit UMKM biasa',
 '\\buniversitas\\b|\\bsekolah\\b': 'Institusi pendidikan',
 '\\bterminal\\b|\\bstasiun\\b': 'Fasilitas publik',
 '\\bfood court\\b': 'Kemungkinan entitas agregat',
 '\\bapartment\\b|\\bapartemen\\b': 'Kemungkinan entitas properti',
 '\\bgriya\\b|\\bborma\\b|\\byogya\\b': 'Dapat merujuk jaringan ritel atau nama usaha lokal; perlu verifikasi',
 '\\bconsina\\b|\\barei\\b|\\berigo\\b|\\belizabeth\\b|\\bbrodo\\b|\\bpaberik badjoe\\b|\\bhouse of donatello\\b': 'Brand '
                                                                                                                   'multi-cabang; '
                                                                                                                   'perlu '
                                                                                                                   'verifikasi '
                                                                                                                   'skala',
 '\\bshabu kojo\\b|\\bhachi grill\\b|\\bkorean house\\b|\\bwizzmie\\b': 'Kemungkinan jaringan kuliner; perlu '
                                                                        'verifikasi',
 '\\blog in megastore\\b|\\bdukomsel\\b': 'Ritel elektronik besar; perlu verifikasi',
 '\\boto xpert\\b|\\botoxpert\\b': 'Jaringan bengkel; perlu verifikasi',
 '\\bprama\\b|\\bsogo\\b': 'Kemungkinan jaringan hotel/ritel besar; perlu verifikasi',
 '\\barein\\b': 'Kemungkinan varian penulisan brand multi-cabang; perlu verifikasi'}

# Daftar lama usaha/nama lokal. Ini HANYA petunjuk tambahan dan tidak dapat
# mengalahkan keputusan manual, kategori non-UMKM, atau bukti jaringan besar.
PETUNJUK_NAMA_LOKAL = ['ibu imas',
 'batagor kingsley',
 'batagor riri',
 'batagor h isan',
 'prima rasa',
 'kartika sari',
 'ampera',
 'alas daun',
 'sindang reret',
 'cibiuk',
 'boemi mitoha',
 'dapoer pandan wangi',
 'kopi purnama',
 'amanah jaya',
 'cibaduyut',
 'cigondewah',
 'brownies amanda',
 'kue balok',
 'surabi',
 'seblak sultan',
 'seblak jebred',
 'mie kocok']

In [ ]:
# ============================================================
# FUNGSI UMUM
# ============================================================

def normalisasi_teks(nilai):
    """Normalisasi defensif untuk pencocokan, tanpa mengubah kolom asli."""
    if pd.isna(nilai):
        return ""
    teks = unicodedata.normalize("NFKC", str(nilai)).lower().strip()
    teks = re.sub(r"[\u2018\u2019`´]", "'", teks)
    teks = re.sub(r"[^a-z0-9]+", " ", teks)
    return re.sub(r"\s+", " ", teks).strip()


def _nama_file_inti(nama):
    """
    Mengabaikan suffix salinan seperti (1), (2), atau (1)(1) pada akhir nama file.
    Berguna jika browser/Colab mengganti nama file ketika file yang sama di-upload ulang.
    """
    basis = os.path.basename(nama).lower()
    stem, ekstensi = os.path.splitext(basis)
    stem = re.sub(r"(?:\s*\(\d+\))+$", "", stem)
    stem = re.sub(r"\s+", " ", stem).strip()
    return stem + ekstensi


def _kolom_header(path):
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        header = f.readline()
    delimiter = ";" if header.count(";") > header.count(",") else ","
    kolom = [x.strip().strip('"') for x in header.rstrip("\r\n").split(delimiter)]
    return kolom, delimiter


def cari_file_input(nama_target, jenis):
    """
    Mencari file secara exact; jika tidak ada, mencoba nama yang mempunyai suffix salinan.
    jenis = 'info' atau 'review'. Pemeriksaan kolom dipakai agar file pasangan tidak tertukar.
    """
    path_exact = os.path.join("/content", nama_target)
    if os.path.exists(path_exact):
        return path_exact

    kandidat = []
    target_inti = _nama_file_inti(nama_target)

    if os.path.exists("/content"):
        for nama in os.listdir("/content"):
            path = os.path.join("/content", nama)
            if not os.path.isfile(path):
                continue
            if _nama_file_inti(nama) != target_inti:
                continue

            try:
                kolom, _ = _kolom_header(path)
                set_kolom = set(kolom)
                cocok_info = {"title", "totalScore", "reviewsCount"}.issubset(set_kolom)
                cocok_review = {"title", "text"}.issubset(set_kolom) and "reviewsCount" not in set_kolom

                if jenis == "info" and cocok_info:
                    kandidat.append(path)
                elif jenis == "review" and cocok_review:
                    kandidat.append(path)
            except Exception:
                continue

    if len(kandidat) == 1:
        print(f"Nama file '{nama_target}' tidak ditemukan secara exact.")
        print(f"Digunakan file yang cocok: {os.path.basename(kandidat[0])}")
        return kandidat[0]

    raise FileNotFoundError(
        f"File {jenis} tidak ditemukan atau ambigu: {nama_target}. "
        "Periksa nama file pada CONFIG_RUNS atau jalankan os.listdir('/content')."
    )


def baca_csv_aman(path, run_id, jenis, tanggal_crawl, urutan_run):
    """
    Membaca CSV dengan deteksi delimiter dan mencatat bad line.
    Bad line tidak hilang diam-diam: disimpan ke audit_parser.
    """
    kolom_header, delimiter = _kolom_header(path)
    bad_lines = []

    def pencatat_bad_line(baris):
        bad_lines.append(baris)
        return None

    df = pd.read_csv(
        path,
        sep=delimiter,
        encoding="utf-8-sig",
        engine="python",
        on_bad_lines=pencatat_bad_line
    )

    df["_source_file"] = os.path.basename(path)
    df["_run_id"] = run_id
    df["_run_order"] = urutan_run
    df["_crawl_date"] = tanggal_crawl or ""
    df["_source_type"] = jenis

    audit_bad = pd.DataFrame({
        "source_file": [os.path.basename(path)] * len(bad_lines),
        "run_id": [run_id] * len(bad_lines),
        "jenis": [jenis] * len(bad_lines),
        "raw_bad_line": [" | ".join(map(str, x)) for x in bad_lines]
    })

    return df, audit_bad, delimiter


def ekstrak_judul_dari_url(nilai):
    """Mencoba merekonstruksi judul dari parameter query pada URL Google Maps."""
    if pd.isna(nilai):
        return None
    teks = str(nilai)
    if "google.com/maps/search" not in teks:
        return None
    cocok = re.search(r"query=([^&]+)", teks)
    if not cocok:
        return None
    return unquote(cocok.group(1)).replace("+", " ").strip()


def koreksi_row_shift_satu_kolom(df):
    """
    Koreksi khusus data Dwi.
    City TIDAK dipaksa menjadi Kota Bandung. Setelah koreksi, lokasi harus lolos
    validasi berdasarkan city/street atau masuk Audit_Lokasi.
    """
    df = df.copy()
    cek_skor = pd.to_numeric(df["totalScore"], errors="coerce")
    mask = (
        cek_skor.isna()
        & df["totalScore"].notna()
        & df["totalScore"].astype(str).str.strip().ne("")
    )

    kandidat = df[mask].copy()
    hasil_terkoreksi = []
    gagal = []

    for _, row in kandidat.iterrows():
        hasil = row.copy()
        judul_url = ekstrak_judul_dari_url(row.get("categoryName"))
        if judul_url:
            hasil["title"] = judul_url

        hasil["totalScore"] = row.get("reviewsCount")
        hasil["reviewsCount"] = row.get("street")
        hasil["street"] = row.get("city")
        hasil["city"] = pd.NA
        hasil["categoryName"] = pd.NA

        rating = pd.to_numeric(pd.Series([hasil["totalScore"]]), errors="coerce").iloc[0]
        jumlah = pd.to_numeric(pd.Series([hasil["reviewsCount"]]), errors="coerce").iloc[0]

        if pd.notna(rating) and 1 <= rating <= 5 and pd.notna(jumlah) and jumlah >= 1:
            hasil["_row_shift_status"] = "CORRECTED"
            hasil_terkoreksi.append(hasil)
        else:
            catatan = row.copy()
            catatan["_drop_reason"] = "ROW_SHIFT_UNRECOVERABLE"
            gagal.append(catatan)

    normal = df[~mask].copy()
    normal["_row_shift_status"] = "NOT_DETECTED"

    df_bersih = pd.concat(
        [normal, pd.DataFrame(hasil_terkoreksi)],
        ignore_index=True,
        sort=False
    )

    return (
        df_bersih,
        pd.DataFrame(gagal),
        pd.DataFrame(hasil_terkoreksi)
    )


def token_url(url):
    """Membuat identitas cadangan dari URL jika street kosong."""
    if pd.isna(url) or not str(url).strip():
        return ""
    teks = str(url)

    for pola in [
        r"query_place_id=([^&]+)",
        r"place_id[:=]([^&]+)",
        r"!1s([^!]+)"
    ]:
        cocok = re.search(pola, teks)
        if cocok:
            return unquote(cocok.group(1))

    return hashlib.sha1(teks.encode("utf-8", errors="ignore")).hexdigest()[:16]


POLA_WILAYAH_LUAR = [
    r"kabupaten bandung barat",
    r"bandung barat",
    r"kabupaten bandung",
    r"kota cimahi",
    r"\bcimahi\b",
    r"\bsumedang\b",
    r"\bgarut\b",
    r"\bsubang\b",
    r"\bpurwakarta\b",
    r"\bcianjur\b"
]

CITY_VALID = {
    "kota bandung",
    "bandung",
    "bandung kota",
    "bandung it",
    "bandung jawa barat",
    "bandung kota jawa barat"
}


def status_lokasi(row):
    city = normalisasi_teks(row.get("city"))
    street = normalisasi_teks(row.get("street"))
    gabungan = f"{city} {street}".strip()

    for pola in POLA_WILAYAH_LUAR:
        if re.search(pola, gabungan):
            return "OUTSIDE_KOTA_BANDUNG"

    if city in CITY_VALID or "kota bandung" in gabungan:
        return "VALID_KOTA_BANDUNG"

    if re.search(r"\bbandung\b", gabungan):
        return "VALID_KOTA_BANDUNG"

    return "NEEDS_LOCATION_REVIEW"


def siapkan_master_kebijakan():
    manual_title = {
        normalisasi_teks(k): v
        for k, v in MANUAL_TITLE_DECISIONS.items()
    }
    exact_drop = {
        normalisasi_teks(k): v
        for k, v in EXACT_DROP_LAMA.items()
    }
    kategori_drop = {normalisasi_teks(x) for x in KATEGORI_AUTO_DROP}
    kategori_review = {normalisasi_teks(x) for x in KATEGORI_PERLU_TINJAU}
    petunjuk_lokal = [normalisasi_teks(x) for x in PETUNJUK_NAMA_LOKAL]

    return manual_title, exact_drop, kategori_drop, kategori_review, petunjuk_lokal


MANUAL_TITLE_NORM, EXACT_DROP_NORM, KATEGORI_DROP_NORM, KATEGORI_REVIEW_NORM, PETUNJUK_LOKAL_NORM = siapkan_master_kebijakan()


def keputusan_entitas(row):
    """
    Menghasilkan:
    entity_decision: KEEP / DROP / REVIEW
    entity_rule
    entity_reason
    entity_source

    Urutan prioritas:
    override entity_key -> keputusan manual title -> exact lama -> pola kuat ->
    kategori auto-drop -> pola/kategori review -> petunjuk lokal -> keep.
    """
    entity_key = str(row.get("entity_key", ""))
    title = normalisasi_teks(row.get("title"))
    kategori = normalisasi_teks(row.get("categoryName"))

    override = MANUAL_ENTITY_OVERRIDES.get(entity_key)
    if override:
        status = override.get("status", "PERLU_TINJAU")
        if status == "UMKM":
            return pd.Series(["KEEP", "ENTITY_MANUAL_KEEP", override.get("alasan", ""), override.get("sumber", "")])
        if status == "BUKAN_UMKM":
            return pd.Series(["DROP", "ENTITY_MANUAL_DROP", override.get("alasan", ""), override.get("sumber", "")])
        return pd.Series(["REVIEW", "ENTITY_MANUAL_REVIEW", override.get("alasan", ""), override.get("sumber", "")])

    manual = MANUAL_TITLE_NORM.get(title)
    if manual:
        status = manual.get("status", "PERLU_TINJAU")
        if status == "UMKM":
            return pd.Series(["KEEP", "TITLE_MANUAL_KEEP", manual.get("alasan", ""), manual.get("sumber", "")])
        if status == "BUKAN_UMKM":
            return pd.Series(["DROP", "TITLE_MANUAL_DROP", manual.get("alasan", ""), manual.get("sumber", "")])
        return pd.Series(["REVIEW", "TITLE_MANUAL_REVIEW", manual.get("alasan", ""), manual.get("sumber", "")])

    if title in EXACT_DROP_NORM:
        return pd.Series(["DROP", "EXACT_CONFIRMED", EXACT_DROP_NORM[title], "Daftar notebook lama"])

    for pola, alasan in POLA_AUTO_DROP.items():
        if re.search(pola, title, flags=re.IGNORECASE):
            return pd.Series(["DROP", "NETWORK_PATTERN", alasan, "Master blacklist"])

    if kategori in KATEGORI_DROP_NORM:
        return pd.Series([
            "DROP",
            "CATEGORY_NON_UMKM",
            f"Kategori '{row.get('categoryName')}' merupakan institusi/entitas agregat, bukan unit usaha lokal tunggal.",
            "Kebijakan operasional kelompok"
        ])

    alasan_review = []

    if kategori in KATEGORI_REVIEW_NORM:
        alasan_review.append(f"Kategori perlu tinjau: {row.get('categoryName')}")

    for pola, alasan in POLA_PERLU_TINJAU.items():
        if re.search(pola, title, flags=re.IGNORECASE):
            alasan_review.append(alasan)

    if alasan_review:
        alasan_unik = "; ".join(dict.fromkeys(alasan_review))
        return pd.Series(["REVIEW", "RULE_REVIEW", alasan_unik, "Aturan audit kandidat"])

    if any(re.search(rf"\b{re.escape(petunjuk)}\b", title) for petunjuk in PETUNJUK_LOKAL_NORM if petunjuk):
        return pd.Series([
            "KEEP",
            "LOCAL_HINT_ONLY",
            "Nama terdapat dalam daftar petunjuk lokal. Ini bukan whitelist mutlak dan tetap dapat ditinjau ulang.",
            "Daftar lama yang direvisi"
        ])

    return pd.Series(["KEEP", "NO_FLAG", "Tidak terkena aturan otomatis.", ""])


def dataframe_aman(df, pesan="Tidak ada data pada kategori ini"):
    if isinstance(df, pd.DataFrame) and not df.empty:
        return df
    return pd.DataFrame({"informasi": [pesan]})

## 1. Load data mentah dan data understanding

In [ ]:
# ============================================================
# LOAD FILE DAN DATA UNDERSTANDING AWAL
# ============================================================

semua_nama_file = []
for run in CONFIG_RUNS:
    semua_nama_file.extend([run["info_file"], run["review_file"]])

file_belum_ada = []
for run in CONFIG_RUNS:
    for nama, jenis in [
        (run["info_file"], "info"),
        (run["review_file"], "review")
    ]:
        try:
            cari_file_input(nama, jenis)
        except FileNotFoundError:
            file_belum_ada.append(nama)

if file_belum_ada:
    print("Upload file berikut:")
    for nama in file_belum_ada:
        print("-", nama)
    print("Notebook dapat mengenali suffix salinan seperti (1) setelah upload.")
    files.upload()

info_runs = []
review_runs = []
audit_parser_list = []
ringkasan_run = []

for urutan, run in enumerate(CONFIG_RUNS, start=1):
    path_info = cari_file_input(run["info_file"], "info")
    path_review = cari_file_input(run["review_file"], "review")

    df_info, bad_info, delimiter_info = baca_csv_aman(
        path_info, run["run_id"], "info", run.get("crawl_date"), urutan
    )
    df_review, bad_review, delimiter_review = baca_csv_aman(
        path_review, run["run_id"], "review", run.get("crawl_date"), urutan
    )

    info_runs.append(df_info)
    review_runs.append(df_review)
    audit_parser_list.extend([bad_info, bad_review])

    ringkasan_run.append({
        "run_id": run["run_id"],
        "crawl_date": run.get("crawl_date") or "Tidak diketahui",
        "info_file": os.path.basename(path_info),
        "review_file": os.path.basename(path_review),
        "delimiter_info": delimiter_info,
        "delimiter_review": delimiter_review,
        "baris_info": len(df_info),
        "baris_review": len(df_review),
        "bad_line_info": len(bad_info),
        "bad_line_review": len(bad_review)
    })

ringkasan_run = pd.DataFrame(ringkasan_run)
audit_parser = pd.concat(audit_parser_list, ignore_index=True) if audit_parser_list else pd.DataFrame()

display(ringkasan_run)

info_mentah = pd.concat(info_runs, ignore_index=True, sort=False)
review_mentah = pd.concat(review_runs, ignore_index=True, sort=False)

KOLOM_INFO_WAJIB = ["title", "totalScore", "reviewsCount", "street", "city", "categoryName"]
KOLOM_REVIEW_WAJIB = ["title", "text"]

for kolom in KOLOM_INFO_WAJIB:
    if kolom not in info_mentah.columns:
        raise KeyError(f"Kolom wajib info tidak ditemukan: {kolom}")

for kolom in KOLOM_REVIEW_WAJIB:
    if kolom not in review_mentah.columns:
        raise KeyError(f"Kolom wajib review tidak ditemukan: {kolom}")

for kolom in ["url"]:
    if kolom not in info_mentah.columns:
        info_mentah[kolom] = pd.NA

for kolom in ["reviewId", "reviewUrl", "stars"]:
    if kolom not in review_mentah.columns:
        review_mentah[kolom] = pd.NA

print("\n=== DATA MENTAH ===")
print(f"Info tempat : {len(info_mentah):,} baris/listing hasil run")
print(f"Review      : {len(review_mentah):,} baris ulasan individual")
print("\nSatuan tersebut tidak boleh dicampur.")

print("\n=== MISSING VALUE KOLOM UTAMA INFO ===")
display(info_mentah[KOLOM_INFO_WAJIB + ["url"]].isna().sum().to_frame("jumlah_missing"))

print("\n=== TOP CITY MENTAH ===")
display(info_mentah["city"].value_counts(dropna=False).head(20).to_frame("jumlah"))

print("\n=== TOP KATEGORI MENTAH ===")
display(info_mentah["categoryName"].value_counts(dropna=False).head(25).to_frame("jumlah"))

## 2. Preprocessing info tempat

In [ ]:
# ============================================================
# PREPROCESSING INFO TEMPAT
# ============================================================

jumlah_info_mentah = len(info_mentah)

# 1. Row-shift
if ENABLE_ROW_SHIFT:
    info_setelah_shift, drop_row_shift, row_shift_terkoreksi = koreksi_row_shift_satu_kolom(info_mentah)
else:
    info_setelah_shift = info_mentah.copy()
    info_setelah_shift["_row_shift_status"] = "NOT_APPLICABLE"
    drop_row_shift = pd.DataFrame()
    row_shift_terkoreksi = pd.DataFrame()

# 2. Validasi title dan numerik
info_setelah_shift["totalScore_num"] = pd.to_numeric(info_setelah_shift["totalScore"], errors="coerce")
info_setelah_shift["reviewsCount_num"] = pd.to_numeric(info_setelah_shift["reviewsCount"], errors="coerce")

kondisi_invalid = [
    info_setelah_shift["title"].isna() | info_setelah_shift["title"].astype(str).str.strip().eq(""),
    info_setelah_shift["totalScore_num"].isna(),
    (info_setelah_shift["totalScore_num"] < 1) | (info_setelah_shift["totalScore_num"] > 5),
    info_setelah_shift["reviewsCount_num"].isna(),
    info_setelah_shift["reviewsCount_num"] < 1
]

alasan_invalid = [
    "MISSING_TITLE",
    "MISSING_OR_NONNUMERIC_RATING",
    "RATING_OUT_OF_RANGE",
    "MISSING_OR_NONNUMERIC_REVIEWCOUNT",
    "REVIEWCOUNT_BELOW_1"
]

info_setelah_shift["_numeric_reason"] = np.select(
    kondisi_invalid,
    alasan_invalid,
    default=""
)

drop_numerik = info_setelah_shift[info_setelah_shift["_numeric_reason"] != ""].copy()
info_valid = info_setelah_shift[info_setelah_shift["_numeric_reason"] == ""].copy()

info_valid["totalScore"] = info_valid["totalScore_num"].astype(float)
info_valid["reviewsCount"] = info_valid["reviewsCount_num"].round().astype("int64")

# 3. Normalisasi identitas
info_valid["title_normalized"] = info_valid["title"].map(normalisasi_teks)
info_valid["street_normalized"] = info_valid["street"].map(normalisasi_teks)
info_valid["city_raw"] = info_valid["city"]
info_valid["city_normalized"] = info_valid["city"].map(normalisasi_teks)
info_valid["place_token"] = info_valid["url"].map(token_url)

info_valid["id_source"] = np.where(
    info_valid["street_normalized"].ne(""),
    "TITLE_STREET",
    np.where(
        info_valid["place_token"].ne(""),
        "TITLE_URL",
        "FALLBACK_AMBIGUOUS"
    )
)

info_valid["entity_key"] = np.where(
    info_valid["street_normalized"].ne(""),
    info_valid["title_normalized"] + " || " + info_valid["street_normalized"],
    np.where(
        info_valid["place_token"].ne(""),
        info_valid["title_normalized"] + " || URL:" + info_valid["place_token"],
        info_valid["title_normalized"]
        + " || FALLBACK:"
        + info_valid["city_normalized"]
        + "|"
        + info_valid["totalScore"].astype(str)
        + "|"
        + info_valid["reviewsCount"].astype(str)
    )
)

audit_street_kosong = info_valid[info_valid["street_normalized"].eq("")].copy()

# Jika street kosong tetapi URL tersedia, listing masih mempunyai identitas cadangan.
# Jika street DAN URL kosong, identitas terlalu ambigu dan dikarantina.
drop_identitas_ambigu = info_valid[info_valid["id_source"].eq("FALLBACK_AMBIGUOUS")].copy()
info_identitas_valid = info_valid[~info_valid["id_source"].eq("FALLBACK_AMBIGUOUS")].copy()

# 4. Deduplikasi: utamakan run terbaru; bila tanggal tidak ada, urutan run lebih besar.
info_identitas_valid["_complete_score"] = (
    info_identitas_valid[["title", "street", "city", "categoryName", "url"]]
    .notna()
    .sum(axis=1)
)
info_identitas_valid["_date_sort"] = pd.to_datetime(
    info_identitas_valid["_crawl_date"],
    errors="coerce"
)

info_identitas_valid = info_identitas_valid.sort_values(
    ["entity_key", "_date_sort", "_run_order", "_complete_score", "reviewsCount"],
    ascending=[True, False, False, False, False],
    na_position="last"
)

mask_duplikat = info_identitas_valid.duplicated("entity_key", keep="first")
drop_duplikat = info_identitas_valid[mask_duplikat].copy()
info_dedup = info_identitas_valid[~mask_duplikat].copy()

# 5. Filter wilayah
info_dedup["location_status"] = info_dedup.apply(status_lokasi, axis=1)
audit_lokasi = info_dedup[info_dedup["location_status"].eq("NEEDS_LOCATION_REVIEW")].copy()
drop_wilayah = info_dedup[~info_dedup["location_status"].eq("VALID_KOTA_BANDUNG")].copy()
info_bandung = info_dedup[info_dedup["location_status"].eq("VALID_KOTA_BANDUNG")].copy()
info_bandung["city"] = "Kota Bandung"

# 6. Audit brand/non-UMKM
hasil_keputusan = info_bandung.apply(keputusan_entitas, axis=1)
hasil_keputusan.columns = [
    "entity_decision",
    "entity_rule",
    "entity_reason",
    "entity_source"
]

info_audit = pd.concat(
    [info_bandung.reset_index(drop=True), hasil_keputusan.reset_index(drop=True)],
    axis=1
)

audit_entitas = info_audit[info_audit["entity_decision"].eq("REVIEW")].copy()
drop_entitas = info_audit[info_audit["entity_decision"].eq("DROP")].copy()

# Dataset calon merger mempertahankan KEEP + REVIEW.
# REVIEW belum boleh langsung dimasukkan ke modeling final sebelum diputuskan kelompok.
info_siap_integrasi = info_audit[~info_audit["entity_decision"].eq("DROP")].copy()

print("=== RINGKASAN INFO TEMPAT ===")
print(f"Mentah                           : {jumlah_info_mentah:,}")
print(f"Setelah row-shift                : {len(info_setelah_shift):,}")
print(f"Setelah validasi title/numerik   : {len(info_valid):,}")
print(f"Setelah identitas ambigu         : {len(info_identitas_valid):,}")
print(f"Setelah deduplikasi              : {len(info_dedup):,}")
print(f"Setelah filter wilayah           : {len(info_bandung):,}")
print(f"Setelah confirmed non-UMKM drop  : {len(info_siap_integrasi):,}")
print(f"Kandidat REVIEW                  : {len(audit_entitas):,}")

## 3. Deduplikasi review dan integrasi teks

In [ ]:
# ============================================================
# CLEANING REVIEW DAN INTEGRASI TEKS
# ============================================================

review = review_mentah.copy()
review["title_normalized"] = review["title"].map(normalisasi_teks)
review["text_original"] = review["text"]
review["text_normalized"] = review["text"].map(normalisasi_teks)
review["reviewId_clean"] = review["reviewId"].fillna("").astype(str).str.strip()
review["stars_num"] = pd.to_numeric(review["stars"], errors="coerce")

mask_review_invalid = (
    review["title_normalized"].eq("")
    | review["text_normalized"].isin(["", "nan", "name"])
    | review["text_normalized"].str.fullmatch(r"[-_ ]*", na=False)
    | (review["stars_num"].notna() & ~review["stars_num"].between(1, 5))
)

drop_review_invalid = review[mask_review_invalid].copy()
review_valid = review[~mask_review_invalid].copy()

# Prioritas reviewId. Jika kosong, fallback title + text.
review_valid["_review_key"] = np.where(
    review_valid["reviewId_clean"].ne(""),
    "ID:" + review_valid["reviewId_clean"],
    "TXT:" + review_valid["title_normalized"] + " || " + review_valid["text_normalized"]
)

review_valid = review_valid.sort_values(
    ["_review_key", "_run_order"],
    ascending=[True, False]
)

mask_review_duplikat = review_valid.duplicated("_review_key", keep="first")
drop_review_duplikat = review_valid[mask_review_duplikat].copy()
review_unik = review_valid[~mask_review_duplikat].copy()

# Review hanya dipetakan jika satu title mempunyai tepat satu listing aktif.
jumlah_listing_per_title = (
    info_siap_integrasi
    .groupby("title_normalized")["entity_key"]
    .nunique()
)

title_unik = set(jumlah_listing_per_title[jumlah_listing_per_title.eq(1)].index)
title_ambigu = set(jumlah_listing_per_title[jumlah_listing_per_title.gt(1)].index)

drop_review_ambigu = review_unik[review_unik["title_normalized"].isin(title_ambigu)].copy()
drop_review_tidak_cocok = review_unik[
    ~review_unik["title_normalized"].isin(title_unik | title_ambigu)
].copy()
review_untuk_join = review_unik[review_unik["title_normalized"].isin(title_unik)].copy()

agregat_review = (
    review_untuk_join
    .groupby("title_normalized")
    .agg(
        text=(
            "text_original",
            lambda nilai: " ||| ".join(
                dict.fromkeys(
                    [
                        str(x).strip()
                        for x in nilai
                        if str(x).strip()
                    ]
                )
            )
        ),
        jumlah_teks_untuk_sentimen=("text_original", "size")
    )
    .reset_index()
)

data_terintegrasi = info_siap_integrasi.merge(
    agregat_review,
    on="title_normalized",
    how="left"
)

mask_tanpa_text = (
    data_terintegrasi["text"].isna()
    | data_terintegrasi["text"].astype(str).str.strip().eq("")
)

data_terintegrasi["_text_drop_reason"] = np.where(
    data_terintegrasi["title_normalized"].isin(title_ambigu),
    "TITLE_MULTI_LISTING_AMBIGUOUS",
    np.where(mask_tanpa_text, "NO_MATCHING_VALID_TEXT", "")
)

drop_tanpa_text = data_terintegrasi[mask_tanpa_text].copy()
data_siap_merger = data_terintegrasi[~mask_tanpa_text].copy()

# Dua keluaran:
# 1. Data_Siap_Merger = KEEP + REVIEW, agar kandidat tidak hilang sebelum keputusan manual.
# 2. Data_Keep_Ketat  = hanya KEEP, aman tetapi lebih konservatif.
data_keep_ketat = data_siap_merger[data_siap_merger["entity_decision"].eq("KEEP")].copy()

kolom_model = [
    "title", "totalScore", "reviewsCount",
    "street", "city", "categoryName", "text"
]
data_model_7kolom = data_keep_ketat[kolom_model].reset_index(drop=True)

print("=== RINGKASAN REVIEW ===")
print(f"Review mentah                    : {len(review_mentah):,}")
print(f"Review invalid/row rusak         : {len(drop_review_invalid):,}")
print(f"Review duplikat                  : {len(drop_review_duplikat):,}")
print(f"Review unik                      : {len(review_unik):,}")
print(f"Review title tidak cocok info    : {len(drop_review_tidak_cocok):,}")
print(f"Title multi-listing ambigu       : {len(title_ambigu):,}")
print()
print(f"Data siap merger (KEEP + REVIEW) : {len(data_siap_merger):,}")
print(f"Data ketat (KEEP saja)           : {len(data_keep_ketat):,}")
print(f"Kandidat REVIEW tersisa          : {(data_siap_merger['entity_decision'] == 'REVIEW').sum():,}")

## 4. Export dan tabel rekonsiliasi

Workbook menyimpan data siap merger, data konservatif `KEEP`, serta seluruh audit/drop. Kandidat `REVIEW` tidak boleh langsung masuk NLP final sebelum diputuskan kelompok.

In [ ]:
# ============================================================
# VALIDASI, REKONSILIASI, DAN EXPORT
# ============================================================

# Validasi dasar
assert data_model_7kolom["totalScore"].between(1, 5).all()
assert data_model_7kolom["reviewsCount"].ge(1).all()
assert data_model_7kolom["text"].notna().all()
assert not data_model_7kolom["text"].astype(str).str.strip().eq("").any()

kolom_audit = [
    "run_id", "crawl_date", "urutan_run",
    "entity_key", "review_key",
    "title", "totalScore", "reviewsCount", "street", "city", "categoryName", "url",
    "reviewId", "reviewUrl", "stars", "text",
    "status_lokasi",
    "entity_decision", "entity_rule", "entity_reason", "entity_source",
    "review_decision", "review_rule", "review_reason"
]

output_csv_base = f"01_Hasil_Preprocessing_{NAMA_ANGGOTA}_Final"

# Export dataset utama ke CSV
data_model_7kolom.to_csv(
    OUTPUT_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

# Export file-file audit ke CSV
data_siap_merger[kolom_audit].to_csv(f"{output_csv_base}_Data_Siap_Merger.csv", index=False, sep=";", encoding="utf-8-sig")
data_keep_ketat[kolom_audit].to_csv(f"{output_csv_base}_Data_Keep_Ketat.csv", index=False, sep=";", encoding="utf-8-sig")
ringkasan_tahap.to_csv(f"{output_csv_base}_Ringkasan_Tahap.csv", index=False, sep=";", encoding="utf-8-sig")
ringkasan_run.to_csv(f"{output_csv_base}_Ringkasan_Run.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(audit_entitas).to_csv(f"{output_csv_base}_Audit_Entitas.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(drop_entitas).to_csv(f"{output_csv_base}_Drop_Entitas.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(audit_lokasi).to_csv(f"{output_csv_base}_Audit_Lokasi.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(drop_wilayah).to_csv(f"{output_csv_base}_Drop_Wilayah.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(audit_street_kosong).to_csv(f"{output_csv_base}_Audit_Street_Kosong.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(drop_identitas_ambigu).to_csv(f"{output_csv_base}_Drop_Identitas.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(drop_duplikat).to_csv(f"{output_csv_base}_Drop_Duplikat.csv", index=False, sep=";", encoding="utf-8-sig")
dataframe_aman(drop_numerik).to_csv(f"{output_csv_base}_Drop_Numerik.csv", index=False, sep=";", encoding="utf-8-sig")

print("Berhasil mengespor semua dataset hasil preprocessing ke format CSV.")
